In [1]:
import numpy as np
import pandas as pd
import pickle

from tensorflow.keras.models import load_model

I0000 00:00:1789497539.196031   20893 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789497539.341198   20893 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789497543.595129   20893 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
model = load_model("model.keras")
with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

E0000 00:00:1789497548.298767   20893 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
df = pd.read_csv("Dataset/AAPL_2006-01-01_to_2018-01-01.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")
df = df.dropna()
df = df.reset_index(drop=True)
print(df.tail())

           Date    Open    High     Low   Close    Volume  Name
3014 2017-12-22  174.68  175.42  174.50  175.01  16349444  AAPL
3015 2017-12-26  170.80  171.47  169.68  170.57  33185536  AAPL
3016 2017-12-27  170.10  170.78  169.71  170.60  21498213  AAPL
3017 2017-12-28  171.00  171.85  170.48  171.08  16480187  AAPL
3018 2017-12-29  170.52  170.59  169.22  169.23  25999922  AAPL


In [4]:
close_prices = df[["Close"]].values
print("Number of closing prices:", len(close_prices))

Number of closing prices: 3019


In [5]:
scaled_prices = scaler.transform(close_prices)

/home/anas/LEARNING/GEN-AI/python/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [6]:
SEQUENCE_LENGTH = 60
last_60_days = scaled_prices[-SEQUENCE_LENGTH:]
print("Last 60 days shape:", last_60_days.shape)

Last 60 days shape: (60, 1)


In [7]:
X_input = np.array([last_60_days])
print("LSTM input shape:", X_input.shape)

LSTM input shape: (1, 60, 1)


In [8]:
prediction_scaled = model.predict(X_input)
print("Scaled prediction:", prediction_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step
Scaled prediction: [[1.2702284]]


In [9]:
prediction = scaler.inverse_transform(prediction_scaled)
predicted_price = prediction[0][0]
print(f"Predicted next AAPL closing price: ${predicted_price:.2f}")

Predicted next AAPL closing price: $166.98


In [10]:
latest_actual_price = df["Close"].iloc[-1]
print(f"Latest available AAPL closing price: " f"${latest_actual_price:.2f}")
print(f"Predicted next AAPL closing price: " f"${predicted_price:.2f}")

Latest available AAPL closing price: $169.23
Predicted next AAPL closing price: $166.98
